# LeetCode 1385 — Find the Distance Value Between Two Arrays bằng Python

**Link đề:** https://leetcode.com/problems/find-the-distance-value-between-two-arrays/
**Độ khó:** Easy · **Chủ đề:** Array, Two Pointers, Binary Search, Sorting

```python
class Solution:
    def findTheDistanceValue(self, arr1: list[int], arr2: list[int], d: int) -> int:
        ...
```

### Mục tiêu học
1. Đọc đúng điều kiện "xa" (far): **mọi** `arr2[j]` thoả `|arr1[i] - arr2[j]| > d` — và biến đổi tương đương: **không có** `arr2[j]` nào nằm trong đoạn `[arr1[i] - d, arr1[i] + d]`.
2. Dùng `abs`, `all`, `any` và hiểu **De Morgan** cho "mọi" / "tồn tại".
3. Từ vét cạn `O(n·m)` → **sắp xếp + binary search** `O((n + m) log m)` → **hai con trỏ** sau khi sắp xếp cả hai mảng.
4. Tự viết binary search tìm "phần tử đầu tiên `>= x`" và hiểu `bisect_left`.
5. Trace, chứng minh, phân tích Time/Space, test đối chiếu và tự giải biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho hai mảng số nguyên `arr1`, `arr2` và số nguyên `d`. Phần tử `arr1[i]` gọi là **xa** (far) nếu **mọi** phần tử `arr2[j]` thoả `|arr1[i] - arr2[j]| > d` — nói cách khác, **không có** phần tử nào của `arr2` cách nó **trong phạm vi `d`**.

Trả về **distance value** = số phần tử xa trong `arr1`.

| arr1 | arr2 | d | Output |
|---|---|---|---|
| `[4,5,8]` | `[10,9,1,8]` | 2 | 2 |
| `[1,4,2,3]` | `[-4,-3,6,10,20,30]` | 3 | 2 |
| `[2,1,100,3]` | `[-5,-2,10,-3,7]` | 6 | 1 |

Ví dụ 1: `4` xa (gần nhất là `1`, `|4-1|=3 > 2`); `5` xa (gần nhất là `8`, `|5-8|=3`); `8` không xa (`|8-8|=0 <= 2`).

**Constraints:** `1 <= arr1.length, arr2.length <= 500`; `-1000 <= arr1[i], arr2[j] <= 1000`; `0 <= d <= 100`.

## 2. Nền tảng: `abs`, khoảng cách và đoạn `[x - d, x + d]`

`abs(v)` là **giá trị tuyệt đối**: `abs(-3) == 3`. `|x - y|` là **khoảng cách** giữa hai điểm trên trục số.

**Biến đổi quan trọng:** `|x - y| <= d` ⇔ `-d <= x - y <= d` ⇔ `x - d <= y <= x + d`. Tức `y` **nằm trong đoạn** `[x - d, x + d]`.

Vậy `x` **không xa** ⇔ có ít nhất một `y` trong `arr2` thuộc đoạn `[x - d, x + d]`.

In [1]:
x, d = 5, 2
for y in [1, 3, 7, 8, 10]:
    print(f"y={y:>2}: |x-y|={abs(x - y)} <= d? {abs(x - y) <= d!s:<5} | {x-d} <= y <= {x+d}? {x - d <= y <= x + d}")

y= 1: |x-y|=4 <= d? False | 3 <= y <= 7? False
y= 3: |x-y|=2 <= d? True  | 3 <= y <= 7? True
y= 7: |x-y|=2 <= d? True  | 3 <= y <= 7? True
y= 8: |x-y|=3 <= d? False | 3 <= y <= 7? False
y=10: |x-y|=5 <= d? False | 3 <= y <= 7? False


Hai cột luôn giống nhau — minh hoạ sự tương đương. Viết `x - d <= y <= x + d` là **so sánh chuỗi** của Python (`a <= b <= c` ⇔ `a <= b and b <= c`).

### 2.1. `all`, `any` và De Morgan

- `all(điều kiện cho mọi phần tử)` — `True` nếu **mọi** phần tử thoả.
- `any(...)` — `True` nếu **có ít nhất một** phần tử thoả.
- **De Morgan:** "mọi `y` thoả `|x-y| > d`" ⇔ "**không** tồn tại `y` với `|x-y| <= d`": `all(abs(x-y) > d for y in arr2) == (not any(abs(x-y) <= d for y in arr2))`.

In [2]:
arr2 = [10, 9, 1, 8]
for x in [4, 5, 8]:
    a = all(abs(x - y) > 2 for y in arr2)
    b_ = not any(abs(x - y) <= 2 for y in arr2)
    print(f"x={x}: all(...)={a}, not any(...)={b_}")

x=4: all(...)=True, not any(...)=True
x=5: all(...)=True, not any(...)=True
x=8: all(...)=False, not any(...)=False


`all`/`any` **dừng sớm** (short-circuit): `any` dừng ngay khi gặp phần tử đúng đầu tiên; `all` dừng ngay khi gặp phần tử sai đầu tiên.

## 3. Cách 1 — Vét cạn

Với mỗi `x` trong `arr1`, kiểm tra mọi `y` trong `arr2`.

In [3]:
def distance_brute(arr1, arr2, d):
    count = 0
    for x in arr1:
        far = True
        for y in arr2:
            if abs(x - y) <= d:
                far = False
                break              # đã thấy một y gần → x không xa, dừng sớm
        if far:
            count += 1
    return count

print(distance_brute([4, 5, 8], [10, 9, 1, 8], 2))              # 2
print(distance_brute([1, 4, 2, 3], [-4, -3, 6, 10, 20, 30], 3)) # 2
print(distance_brute([2, 1, 100, 3], [-5, -2, 10, -3, 7], 6))   # 1

2
2
1


In [4]:
def distance_brute_short(arr1, arr2, d):
    return sum(all(abs(x - y) > d for y in arr2) for x in arr1)

print(distance_brute_short([4, 5, 8], [10, 9, 1, 8], 2))

2


`sum(...)` của các giá trị `True/False` = số giá trị `True`.

### Độ phức tạp và nút thắt
- Thời gian: `O(n · m)` với `n = len(arr1)`, `m = len(arr2)`. Với `500 × 500 = 250.000` — vẫn qua.
- **Nút thắt:** với mỗi `x`, ta xét **toàn bộ** `arr2`, dù chỉ cần biết **phần tử gần `x` nhất**. Nếu `arr2` được **sắp xếp**, phần tử gần nhất nằm ngay quanh vị trí mà `x` sẽ được "chèn vào".

## 4. Insight — chỉ cần các "hàng xóm" của `x` trong `arr2` đã sắp

Sắp `arr2` tăng dần. Hỏi: có `y` nào trong `[x - d, x + d]` không?

Tìm `k` = chỉ số **đầu tiên** có `arr2[k] >= x - d`. Khi đó:
- Mọi phần tử trước `k` đều `< x - d` → nằm **ngoài** đoạn (bên trái).
- `arr2[k]` là phần tử **nhỏ nhất** `>= x - d`. Nếu `arr2[k] <= x + d` → nó nằm **trong** đoạn → `x` không xa. Nếu `arr2[k] > x + d` (hoặc `k == m`, không có) → mọi phần tử từ `k` trở đi đều `> x + d` → `x` xa.

Chỉ **một** phép so sánh sau khi tìm `k`. Tìm `k` bằng **binary search** `O(log m)`.

In [5]:
s2 = sorted([10, 9, 1, 8])
print("arr2 đã sắp:", s2)
for x in [4, 5, 8]:
    lo_bound = x - 2
    k = next((i for i, y in enumerate(s2) if y >= lo_bound), len(s2))
    nb = s2[k] if k < len(s2) else None
    print(f"x={x}: đoạn [{x-2}, {x+2}], k={k}, s2[k]={nb} → {'không xa' if nb is not None and nb <= x + 2 else 'xa'}")

arr2 đã sắp: [1, 8, 9, 10]
x=4: đoạn [2, 6], k=1, s2[k]=8 → xa
x=5: đoạn [3, 7], k=1, s2[k]=8 → xa
x=8: đoạn [6, 10], k=1, s2[k]=8 → không xa


(Ô trên tìm `k` bằng duyệt tuyến tính chỉ để minh hoạ ý tưởng; `next(generator, default)` trả `default` nếu generator rỗng.)

## 5. Binary search: phần tử đầu tiên `>= target`

Giữ khoảng `[lo, hi)`, `lo = 0`, `hi = m` (đáp án có thể là `m` = "không có").
- `mid = (lo + hi) // 2`.
- `a[mid] >= target` → đáp án ở `mid` hoặc bên trái → `hi = mid`.
- `a[mid] < target` → đáp án ở bên phải `mid` → `lo = mid + 1`.
- Dừng khi `lo == hi` → trả `lo`.

Đây đúng là `bisect.bisect_left(a, target)` của thư viện chuẩn.

In [6]:
from bisect import bisect_left

def lower_bound(a, target):
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] >= target:
            hi = mid
        else:
            lo = mid + 1
    return lo

a = [1, 8, 9, 10]
for t in [0, 1, 2, 8, 9, 11]:
    print(f"target={t:>2}: lower_bound={lower_bound(a, t)}, bisect_left={bisect_left(a, t)}")

target= 0: lower_bound=0, bisect_left=0
target= 1: lower_bound=0, bisect_left=0
target= 2: lower_bound=1, bisect_left=1
target= 8: lower_bound=1, bisect_left=1
target= 9: lower_bound=2, bisect_left=2
target=11: lower_bound=4, bisect_left=4


In [7]:
def trace_lower_bound(a, target):
    lo, hi = 0, len(a)
    print(f"Tìm phần tử đầu tiên >= {target} trong {a}")
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] >= target:
            print(f"  lo={lo}, hi={hi}, mid={mid}, a[mid]={a[mid]} >= {target} → hi = {mid}")
            hi = mid
        else:
            print(f"  lo={lo}, hi={hi}, mid={mid}, a[mid]={a[mid]} <  {target} → lo = {mid + 1}")
            lo = mid + 1
    print("  → k =", lo)
    return lo

trace_lower_bound([-4, -3, 6, 10, 20, 30], 4 - 3)

Tìm phần tử đầu tiên >= 1 trong [-4, -3, 6, 10, 20, 30]
  lo=0, hi=6, mid=3, a[mid]=10 >= 1 → hi = 3
  lo=0, hi=3, mid=1, a[mid]=-3 <  1 → lo = 2
  lo=2, hi=3, mid=2, a[mid]=6 >= 1 → hi = 2
  → k = 2


2

## 6. Cách 2 (chính) — Sắp xếp `arr2` + binary search

In [8]:
def distance_binary(arr1, arr2, d):
    s = sorted(arr2)
    m = len(s)
    count = 0
    for x in arr1:
        k = lower_bound(s, x - d)
        if k == m or s[k] > x + d:     # không có phần tử nào trong [x-d, x+d]
            count += 1
    return count

print(distance_binary([4, 5, 8], [10, 9, 1, 8], 2))
print(distance_binary([1, 4, 2, 3], [-4, -3, 6, 10, 20, 30], 3))
print(distance_binary([2, 1, 100, 3], [-5, -2, 10, -3, 7], 6))

2
2
1


Thứ tự điều kiện `k == m or s[k] > x + d` **quan trọng**: nếu `k == m` thì `s[k]` vượt chỉ số. Nhờ `or` **dừng sớm**, `s[k]` chỉ được đọc khi `k < m`.

## 7. TRACE — Cách 2 trên ví dụ 2

`arr2` sắp: `[-4, -3, 6, 10, 20, 30]`, `d = 3`.

| x | đoạn `[x-3, x+3]` | k = lower_bound(x-3) | s[k] | s[k] <= x+3? | Kết luận |
|---|---|---|---|---|---|
| 1 | [-2, 4] | 2 | 6 | 6 > 4 | **xa** |
| 4 | [1, 7] | 2 | 6 | 6 <= 7 | không xa |
| 2 | [-1, 5] | 2 | 6 | 6 > 5 | **xa** |
| 3 | [0, 6] | 2 | 6 | 6 <= 6 | không xa |

Đáp án: 2.

In [9]:
def trace_distance_binary(arr1, arr2, d):
    s = sorted(arr2)
    print("arr2 đã sắp:", s, "| d =", d)
    count = 0
    for x in arr1:
        k = lower_bound(s, x - d)
        val = s[k] if k < len(s) else "—"
        far = k == len(s) or s[k] > x + d
        count += far
        print(f"  x={x:>4}: đoạn [{x-d:>4}, {x+d:>4}] | k={k} | s[k]={val!s:>4} | {'XA' if far else 'không xa'} | count={count}")
    return count

trace_distance_binary([1, 4, 2, 3], [-4, -3, 6, 10, 20, 30], 3)
print()
trace_distance_binary([2, 1, 100, 3], [-5, -2, 10, -3, 7], 6)

arr2 đã sắp: [-4, -3, 6, 10, 20, 30] | d = 3
  x=   1: đoạn [  -2,    4] | k=2 | s[k]=   6 | XA | count=1
  x=   4: đoạn [   1,    7] | k=2 | s[k]=   6 | không xa | count=1
  x=   2: đoạn [  -1,    5] | k=2 | s[k]=   6 | XA | count=2
  x=   3: đoạn [   0,    6] | k=2 | s[k]=   6 | không xa | count=2

arr2 đã sắp: [-5, -3, -2, 7, 10] | d = 6
  x=   2: đoạn [  -4,    8] | k=1 | s[k]=  -3 | không xa | count=0
  x=   1: đoạn [  -5,    7] | k=0 | s[k]=  -5 | không xa | count=0
  x= 100: đoạn [  94,  106] | k=5 | s[k]=   — | XA | count=1
  x=   3: đoạn [  -3,    9] | k=1 | s[k]=  -3 | không xa | count=1


1

Ví dụ 3 có nhánh `k == len(s)`: với `x = 100`, đoạn `[94, 106]` nằm bên phải mọi phần tử → `k = 5 = len(s)` → xa.

`count += far` — cộng `True` (=1) hoặc `False` (=0).

## 8. Cách 3 — Sắp cả hai mảng + hai con trỏ

Nếu `arr1` cũng được sắp tăng dần thì `x - d` tăng dần → chỉ số `k` (đầu tiên `>= x - d`) cũng **chỉ tăng**. Thay vì binary search lại từ đầu, ta **giữ** con trỏ `j` và chỉ đẩy nó tiến lên.

```text
j = 0
for x in sorted(arr1):
    while j < m and s2[j] < x - d:  j += 1     # bỏ các phần tử quá nhỏ
    nếu j == m hoặc s2[j] > x + d: x xa
```

Con trỏ `j` đi tổng cộng tối đa `m` bước cho cả vòng lặp → phần duyệt là `O(n + m)`.

(Thứ tự trong `arr1` không ảnh hưởng đáp án vì ta chỉ **đếm** — được phép sắp `arr1`.)

In [10]:
def distance_two_pointers(arr1, arr2, d):
    s1, s2 = sorted(arr1), sorted(arr2)
    m = len(s2)
    j = 0
    count = 0
    for x in s1:
        while j < m and s2[j] < x - d:
            j += 1
        if j == m or s2[j] > x + d:
            count += 1
    return count

print(distance_two_pointers([4, 5, 8], [10, 9, 1, 8], 2))
print(distance_two_pointers([1, 4, 2, 3], [-4, -3, 6, 10, 20, 30], 3))
print(distance_two_pointers([2, 1, 100, 3], [-5, -2, 10, -3, 7], 6))

2
2
1


In [11]:
def trace_two_pointers(arr1, arr2, d):
    s1, s2 = sorted(arr1), sorted(arr2)
    print("s1 =", s1, "| s2 =", s2, "| d =", d)
    j, count = 0, 0
    for x in s1:
        start = j
        while j < len(s2) and s2[j] < x - d:
            j += 1
        val = s2[j] if j < len(s2) else "—"
        far = j == len(s2) or s2[j] > x + d
        count += far
        print(f"  x={x:>3}: j {start}→{j} (bỏ các s2 < {x-d}) | s2[j]={val!s:>3} | {'XA' if far else 'không xa'}")
    print("Kết quả:", count)
    return count

trace_two_pointers([4, 5, 8], [10, 9, 1, 8], 2)

s1 = [4, 5, 8] | s2 = [1, 8, 9, 10] | d = 2
  x=  4: j 0→1 (bỏ các s2 < 2) | s2[j]=  8 | XA
  x=  5: j 1→1 (bỏ các s2 < 3) | s2[j]=  8 | XA
  x=  8: j 1→1 (bỏ các s2 < 6) | s2[j]=  8 | không xa
Kết quả: 2


2

## 9. Cách 4 — Đánh dấu trên miền giá trị nhỏ (tham khảo)

Giá trị chỉ trong `[-1000, 1000]`. Ta có thể làm mảng **prefix count** trên miền: `pre[v]` = số phần tử `arr2` có giá trị `< v` (dịch chỉ số `+1000` để không âm). Số phần tử trong `[x-d, x+d]` = `pre[x+d+1] - pre[x-d]` → `O(1)` mỗi truy vấn, tổng `O(n + m + V)`.

Cần **kẹp** (clamp) biên đoạn về `[-1000, 1000]` vì `x ± d` có thể vượt miền.

In [12]:
def distance_prefix(arr1, arr2, d):
    OFF, V = 1000, 2001                    # giá trị v ↦ chỉ số v + OFF trong [0, 2000]
    cnt = [0] * V
    for y in arr2:
        cnt[y + OFF] += 1
    pre = [0] * (V + 1)                    # pre[k] = số phần tử có chỉ số < k
    for k in range(V):
        pre[k + 1] = pre[k] + cnt[k]
    count = 0
    for x in arr1:
        lo = max(x - d, -1000) + OFF
        hi = min(x + d, 1000) + OFF
        inside = pre[hi + 1] - pre[lo] if lo <= hi else 0
        if inside == 0:
            count += 1
    return count

print(distance_prefix([4, 5, 8], [10, 9, 1, 8], 2), distance_prefix([2, 1, 100, 3], [-5, -2, 10, -3, 7], 6))

2 1


## 10. Code nộp LeetCode

Bản sort + binary search (dùng `bisect_left` cho gọn):

In [13]:
from bisect import bisect_left

class Solution:
    def findTheDistanceValue(self, arr1: list[int], arr2: list[int], d: int) -> int:
        arr2 = sorted(arr2)
        m = len(arr2)
        count = 0
        for x in arr1:
            k = bisect_left(arr2, x - d)
            if k == m or arr2[k] > x + d:
                count += 1
        return count

sol = Solution()
print(sol.findTheDistanceValue([4,5,8], [10,9,1,8], 2))                # 2
print(sol.findTheDistanceValue([1,4,2,3], [-4,-3,6,10,20,30], 3))      # 2
print(sol.findTheDistanceValue([2,1,100,3], [-5,-2,10,-3,7], 6))       # 1

2
2
1


Bản hai con trỏ:

In [14]:
class SolutionTwoPointers:
    def findTheDistanceValue(self, arr1: list[int], arr2: list[int], d: int) -> int:
        arr1, arr2 = sorted(arr1), sorted(arr2)
        j = count = 0
        for x in arr1:
            while j < len(arr2) and arr2[j] < x - d:
                j += 1
            if j == len(arr2) or arr2[j] > x + d:
                count += 1
        return count

print(SolutionTwoPointers().findTheDistanceValue([1,4,2,3], [-4,-3,6,10,20,30], 3))

2


`j = count = 0` — **gán chuỗi**: cả hai biến cùng nhận `0` (an toàn với số; cẩn thận với list vì hai tên sẽ trỏ cùng một list).

## 11. Vì sao đúng?

**Cách 2:** gọi `k = lower_bound(s, x - d)`. Theo định nghĩa: `s[i] < x - d` với mọi `i < k`, và `s[i] >= x - d` với mọi `i >= k` (mảng tăng). Đoạn `[x-d, x+d]` chứa phần tử nào đó ⇔ có `i >= k` với `s[i] <= x + d` ⇔ (vì `s[k]` nhỏ nhất trong các `s[i]`, `i >= k`) `k < m` và `s[k] <= x + d`. Phủ định lại chính là điều kiện "xa" trong code.

**Cách 3 — invariant:** trước khi xét `x` (theo thứ tự tăng), mọi `s2[i]` với `i < j` đều `< x' - d` với `x'` là phần tử trước đó, nên cũng `< x - d` (vì `x >= x'`). Vòng `while` đẩy `j` tới đúng `lower_bound(s2, x - d)`. Từ đó lập luận như Cách 2.

### 11.1. Phân tích Time / Space

Gọi `n = len(arr1)`, `m = len(arr2)`, `V = 2001` (miền giá trị).

| Cách | Thời gian | Bộ nhớ phụ |
|---|---|---|
| 1. Vét cạn | `O(n · m)` | `O(1)` |
| 2. Sort arr2 + binary search | `O(m log m + n log m)` | `O(m)` (bản sắp) |
| 3. Sort cả hai + hai con trỏ | `O(n log n + m log m)` | `O(n + m)` |
| 4. Prefix trên miền giá trị | `O(n + m + V)` | `O(V)` |

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [15]:
import random

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.findTheDistanceValue([4,5,8], [10,9,1,8], 2), 2)
check("đề 2", sol.findTheDistanceValue([1,4,2,3], [-4,-3,6,10,20,30], 3), 2)
check("đề 3", sol.findTheDistanceValue([2,1,100,3], [-5,-2,10,-3,7], 6), 1)
check("d = 0, trùng giá trị", sol.findTheDistanceValue([1, 2, 3], [2], 0), 2)
check("đúng biên |x-y| = d (không xa)", sol.findTheDistanceValue([5], [8], 3), 0)
check("vượt biên 1 (xa)", sol.findTheDistanceValue([5], [9], 3), 1)
check("biên miền giá trị", sol.findTheDistanceValue([-1000, 1000], [0], 100), 2)
check("mọi x đều gần", sol.findTheDistanceValue([1, 1, 1], [1], 0), 0)
check("phần tử trùng trong arr1", sol.findTheDistanceValue([7, 7, 0], [3], 3), 2)

PASS đề 1: 2
PASS đề 2: 2
PASS đề 3: 1
PASS d = 0, trùng giá trị: 2
PASS đúng biên |x-y| = d (không xa): 0
PASS vượt biên 1 (xa): 1
PASS biên miền giá trị: 2
PASS mọi x đều gần: 0
PASS phần tử trùng trong arr1: 2


In [16]:
methods = [distance_brute, distance_brute_short, distance_binary, distance_two_pointers, distance_prefix,
           Solution().findTheDistanceValue, SolutionTwoPointers().findTheDistanceValue]
rng = random.Random(1385)
for _ in range(500):
    span = rng.choice([5, 50, 1000])
    a1 = [rng.randint(-span, span) for _ in range(rng.randint(1, 30))]
    a2 = [rng.randint(-span, span) for _ in range(rng.randint(1, 30))]
    dd = rng.randint(0, 100)
    expected = distance_brute(a1, a2, dd)
    for f in methods:
        assert f(list(a1), list(a2), dd) == expected, (f, a1, a2, dd)
print(f"500 bộ ngẫu nhiên × {len(methods)} cách: khớp nhau — PASS")

500 bộ ngẫu nhiên × 7 cách: khớp nhau — PASS


Ta gọi `f(list(a1), list(a2), dd)` với **bản sao** để nếu một hàm lỡ sửa mảng đầu vào thì các hàm sau vẫn nhận dữ liệu gốc.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng `>=` thay vì `>` (hoặc `<` thay vì `<=`)
Đề: xa khi `|x - y| > d` **nghiêm ngặt**. `|x - y| == d` là **không xa**.

In [17]:
def wrong_ge(arr1, arr2, d):
    return sum(all(abs(x - y) >= d for y in arr2) for x in arr1)

print("Sai :", wrong_ge([5], [8], 3), "| Đúng:", distance_brute([5], [8], 3))

Sai : 1 | Đúng: 0


### Lỗi 2: Đếm cặp thay vì đếm phần tử
Đếm số cặp `(i, j)` xa nhau thay vì số `x` xa **mọi** `y`.

In [18]:
def wrong_pairs(arr1, arr2, d):
    return sum(1 for x in arr1 for y in arr2 if abs(x - y) > d)

print("Sai :", wrong_pairs([4, 5, 8], [10, 9, 1, 8], 2), "| Đúng:", distance_brute([4, 5, 8], [10, 9, 1, 8], 2))

Sai : 9 | Đúng: 2


### Lỗi 3: Binary search nhưng chỉ kiểm tra **một phía**
Chỉ xét phần tử `>= x` (bỏ quên phần tử nhỏ hơn `x` nhưng vẫn trong `[x-d, x)`).

In [19]:
def wrong_one_side(arr1, arr2, d):
    s = sorted(arr2)
    count = 0
    for x in arr1:
        k = bisect_left(s, x)                    # chỉ tìm phần tử >= x
        if k == len(s) or s[k] - x > d:
            count += 1
    return count

print("Sai :", wrong_one_side([5], [4], 2), "| Đúng:", distance_brute([5], [4], 2))

Sai : 1 | Đúng: 0


Sửa: tìm `lower_bound(x - d)` (như Cách 2), hoặc kiểm tra **cả hai** hàng xóm `s[k-1]` và `s[k]` quanh `x`.

### Lỗi 4: Quên kiểm `k == m` trước khi đọc `s[k]`
Viết `if s[k] > x + d or k == m` → `IndexError` khi `k == m` vì vế trái được tính trước.

In [20]:
def wrong_order(arr1, arr2, d):
    s = sorted(arr2)
    count = 0
    for x in arr1:
        k = bisect_left(s, x - d)
        if s[k] > x + d or k == len(s):          # đọc s[k] trước khi kiểm k
            count += 1
    return count

try:
    wrong_order([100], [1, 2], 5)
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


### Mẹo nhận dạng pattern

- **"Với mỗi x, có phần tử nào của mảng khác nằm trong khoảng ...?"** → sắp mảng kia + **binary search** biên trái của khoảng.
- Cả hai phía đều có thể sắp và chỉ cần **đếm** → **hai con trỏ** đơn điệu.
- `|x - y| <= d` ⇔ `y ∈ [x - d, x + d]` — chuyển trị tuyệt đối thành **đoạn**.
- Miền giá trị nhỏ → **prefix count** trả lời "có bao nhiêu phần tử trong đoạn" `O(1)`.
- Bài anh em: *Find K Closest Elements* (658), *Heaters* (475), *Minimum Absolute Difference* (1200), *Contains Duplicate III* (220).

## 14. Biến thể

### 14.1. Khoảng cách tới phần tử gần nhất
Với mỗi `x`, trả `min |x - y|`. Binary search vị trí chèn `k`, chỉ cần xét `s[k-1]` và `s[k]`.

In [21]:
def nearest_distances(arr1, arr2):
    s = sorted(arr2)
    res = []
    for x in arr1:
        k = bisect_left(s, x)
        best = float("inf")
        if k < len(s):
            best = min(best, s[k] - x)
        if k > 0:
            best = min(best, x - s[k - 1])
        res.append(best)
    return res

print(nearest_distances([4, 5, 8], [10, 9, 1, 8]))   # [3, 3, 0]

[3, 3, 0]


Hệ quả: `x` xa ⇔ `khoảng cách gần nhất > d`. Đây là một cách giải thứ năm cho bài gốc.

### 14.2. Heaters (LeetCode 475) — cùng ý tưởng
Bán kính nhỏ nhất để mọi nhà đều được sưởi = `max` của khoảng cách từ mỗi nhà tới lò sưởi gần nhất.

In [22]:
def find_radius(houses, heaters):
    return max(nearest_distances(houses, heaters))

print(find_radius([1, 2, 3], [2]), find_radius([1, 2, 3, 4], [1, 4]), find_radius([1, 5], [2]))   # 1 1 3

1 1 3


### 14.3. Đếm phần tử của `arr2` nằm trong phạm vi `d` của mỗi `x`
Dùng hai lần binary search: `bisect_right(s, x+d) - bisect_left(s, x-d)`.

In [23]:
from bisect import bisect_right

def count_close(arr1, arr2, d):
    s = sorted(arr2)
    return [bisect_right(s, x + d) - bisect_left(s, x - d) for x in arr1]

print(count_close([4, 5, 8], [10, 9, 1, 8], 2))   # [0, 0, 3]

[0, 0, 3]


## 15. Bài tập (TODO — đáp án ở cuối)

### Bài 1 — Tính tay
`arr1 = [0, 10, 20]`, `arr2 = [3, 14, 26]`, `d = 4`. Phần tử nào xa? Lập bảng như mục 7.

### Bài 2 — Vét cạn với `any`

In [24]:
def exercise_brute_any(arr1, arr2, d):
    # TODO: đếm x mà not any(|x - y| <= d)
    pass

# Gợi ý test:
# assert exercise_brute_any([4, 5, 8], [10, 9, 1, 8], 2) == 2

### Bài 3 — Tự viết `lower_bound` và dùng nó

In [25]:
def exercise_lower_bound(a, target):
    # TODO: [lo, hi), trả chỉ số đầu tiên a[i] >= target
    pass

def exercise_distance_binary(arr1, arr2, d):
    # TODO: sort arr2, dùng exercise_lower_bound
    pass

# Gợi ý test:
# assert exercise_lower_bound([1, 8, 9, 10], 2) == 1
# assert exercise_lower_bound([1, 8, 9, 10], 11) == 4
# assert exercise_distance_binary([2, 1, 100, 3], [-5, -2, 10, -3, 7], 6) == 1

### Bài 4 — Hai con trỏ

In [26]:
def exercise_two_pointers(arr1, arr2, d):
    # TODO
    pass

# Gợi ý test:
# assert exercise_two_pointers([1, 4, 2, 3], [-4, -3, 6, 10, 20, 30], 3) == 2

### Bài 5 — Trả về danh sách phần tử xa (giữ thứ tự gốc)

In [27]:
def exercise_far_elements(arr1, arr2, d):
    # TODO
    pass

# Gợi ý test:
# assert exercise_far_elements([4, 5, 8], [10, 9, 1, 8], 2) == [4, 5]
# assert exercise_far_elements([2, 1, 100, 3], [-5, -2, 10, -3, 7], 6) == [100]

### Bài 6 — Mini challenge: `d` lớn nhất để tất cả đều xa
Trả về `d` **lớn nhất** (`>= 0`) sao cho distance value bằng `len(arr1)` (mọi phần tử đều xa); nếu không có `d >= 0` nào thoả thì trả `-1`.

In [28]:
def exercise_max_d_all_far(arr1, arr2):
    # TODO: liên hệ với khoảng cách gần nhất
    pass

# Gợi ý test:
# assert exercise_max_d_all_far([4, 5], [10, 9, 1, 8]) == 2
# assert exercise_max_d_all_far([8], [8]) == -1

## 16. Gợi ý 3 mức

**Bài 1** — (1) Đoạn của mỗi x: `[-4,4]`, `[6,14]`, `[16,24]`. (2) Có phần tử `arr2` nào trong đó? (3) `3 ∈ [-4,4]`, `14 ∈ [6,14]`, không có gì trong `[16,24]`.

**Bài 2** — (1) De Morgan. (2) `sum(not any(...) for x in arr1)`. (3) Điều kiện trong `any`: `abs(x - y) <= d`.

**Bài 3** — (1) Mục 5. (2) `if a[mid] >= target: hi = mid else: lo = mid + 1`. (3) Điều kiện xa: `k == m or s[k] > x + d`.

**Bài 4** — (1) Sắp cả hai. (2) `while j < m and s2[j] < x - d: j += 1`. (3) Kiểm như Cách 2.

**Bài 5** — (1) Giống đếm nhưng `append`. (2) Duyệt `arr1` gốc (không sắp) để giữ thứ tự. (3) Dùng binary search trên bản sắp của `arr2`.

**Bài 6** — (1) `x` xa ⇔ `nearest(x) > d`. (2) Mọi `x` xa ⇔ `d < min(nearest(x))`. (3) Đáp án `min(nearest) - 1` (âm thì `-1`).

## 17. Đáp án tham khảo

**Bài 1:** `0` không xa (`3`), `10` không xa (`14`, `|10-14| = 4 <= 4`), `20` xa → đáp án **1**.

In [29]:
assert Solution().findTheDistanceValue([0, 10, 20], [3, 14, 26], 4) == 1

def answer_brute_any(arr1, arr2, d):
    return sum(not any(abs(x - y) <= d for y in arr2) for x in arr1)

def answer_lower_bound(a, target):
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] >= target:
            hi = mid
        else:
            lo = mid + 1
    return lo

def answer_distance_binary(arr1, arr2, d):
    s = sorted(arr2)
    return sum(1 for x in arr1
               if (k := answer_lower_bound(s, x - d)) == len(s) or s[k] > x + d)

def answer_two_pointers(arr1, arr2, d):
    s1, s2 = sorted(arr1), sorted(arr2)
    j = count = 0
    for x in s1:
        while j < len(s2) and s2[j] < x - d:
            j += 1
        if j == len(s2) or s2[j] > x + d:
            count += 1
    return count

def answer_far_elements(arr1, arr2, d):
    s = sorted(arr2)
    res = []
    for x in arr1:
        k = bisect_left(s, x - d)
        if k == len(s) or s[k] > x + d:
            res.append(x)
    return res

def answer_max_d_all_far(arr1, arr2):
    best = min(nearest_distances(arr1, arr2)) - 1
    return best if best >= 0 else -1

assert answer_brute_any([4, 5, 8], [10, 9, 1, 8], 2) == 2
assert answer_lower_bound([1, 8, 9, 10], 2) == 1 and answer_lower_bound([1, 8, 9, 10], 11) == 4
assert answer_distance_binary([2, 1, 100, 3], [-5, -2, 10, -3, 7], 6) == 1
assert answer_two_pointers([1, 4, 2, 3], [-4, -3, 6, 10, 20, 30], 3) == 2
assert answer_far_elements([4, 5, 8], [10, 9, 1, 8], 2) == [4, 5]
assert answer_far_elements([2, 1, 100, 3], [-5, -2, 10, -3, 7], 6) == [100]
assert answer_max_d_all_far([4, 5], [10, 9, 1, 8]) == 2
assert answer_max_d_all_far([8], [8]) == -1

rng = random.Random(11)
for _ in range(300):
    a1 = [rng.randint(-50, 50) for _ in range(rng.randint(1, 15))]
    a2 = [rng.randint(-50, 50) for _ in range(rng.randint(1, 15))]
    dd = rng.randint(0, 20)
    e = distance_brute(a1, a2, dd)
    assert answer_brute_any(a1, a2, dd) == answer_distance_binary(a1, a2, dd) == answer_two_pointers(a1, a2, dd) == e
    assert len(answer_far_elements(a1, a2, dd)) == e
    md = answer_max_d_all_far(a1, a2)
    if md >= 0:
        assert distance_brute(a1, a2, md) == len(a1) and distance_brute(a1, a2, md + 1) < len(a1)
print("Đáp án bài tập: PASS toàn bộ")

Đáp án bài tập: PASS toàn bộ


Trong `answer_distance_binary` có **toán tử walrus** `:=` (Python 3.8+): `(k := biểu_thức)` vừa **gán** vừa **trả** giá trị, giúp dùng `k` ngay trong cùng biểu thức.

## 18. Tổng kết — template nhớ nhanh + checklist

```python
s = sorted(arr2)
for x in arr1:
    k = bisect_left(s, x - d)          # phần tử đầu tiên >= x - d
    if k == len(s) or s[k] > x + d:    # không có gì trong [x-d, x+d]
        count += 1
```

**Checklist trước khi submit**
- [ ] "Xa" là `> d` nghiêm ngặt; `== d` là gần.
- [ ] Đếm **phần tử** `arr1`, không đếm cặp.
- [ ] Binary search biên **trái** của đoạn (`x - d`), không phải `x`.
- [ ] Kiểm `k == len(s)` **trước** khi đọc `s[k]`.
- [ ] Đã test `d = 0`, giá trị trùng, đúng biên `|x - y| = d`, biên miền `±1000`.